In [16]:
!pip install streamlit google-generativeai pyngrok pdfplumber fpdf reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.2 MB/s eta 0:00:00


In [17]:
%%writefile app.py

import streamlit as st
import google.generativeai as genai
import pdfplumber

from report_generator import generate_pdf

# API KEY
genai.configure(api_key="YOUR_GEMINI_API_KEY")

model = genai.GenerativeModel("gemini-1.5-flash")

st.set_page_config(
    page_title="MeetMind AI",
    page_icon="🧠",
    layout="wide"
)

st.title("🧠 MeetMind AI")
st.caption("AI Powered Meeting Intelligence Assistant")

uploaded_file = st.file_uploader(
    "Upload Meeting Transcript",
    type=["pdf","txt"]
)

meeting_notes = st.text_area(
    "Or Paste Meeting Notes",
    height=250
)

text = ""

if uploaded_file:

    if uploaded_file.name.endswith(".pdf"):

        with pdfplumber.open(uploaded_file) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text += page_text

    else:

        text = str(uploaded_file.read(),"utf-8")

if meeting_notes:
    text += meeting_notes

if st.button("Analyze Meeting"):

    if text.strip() == "":
        st.warning("Please upload or enter meeting notes.")
        st.stop()

    prompt = f"""
You are a professional business analyst.

Analyze the meeting transcript.

Return:

1. Executive Summary

2. Key Decisions

3. Action Items

4. Task Owners

5. Deadlines

6. Follow-up Email

Meeting Notes:

{text}
"""

    with st.spinner("Analyzing Meeting..."):

        response = model.generate_content(prompt)

        result = response.text

    st.session_state["report"] = result

    st.markdown(result)

    pdf_file = generate_pdf(result)

    st.download_button(
        label="Download Report",
        data=pdf_file,
        file_name="meeting_report.pdf",
        mime="application/pdf"
    )

Overwriting app.py


In [18]:
%%writefile report_generator.py
from fpdf import FPDF
from io import BytesIO

def generate_pdf(report_content: str) -> BytesIO:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    # Split content by lines to fit within PDF page width
    for line in report_content.split('\n'):
        # Add a line break after each line to ensure proper formatting
        pdf.multi_cell(0, 10, txt=line)

    # Save the PDF to a BytesIO object
    pdf_output = BytesIO()
    # FPDF.output(dest='S') returns the document as a byte string directly
    # The double call to output() and then encode('latin1') is redundant and incorrect.
    # We should just get the byte string once and write it.
    pdf_bytes = pdf.output(dest='S').encode('latin1')
    pdf_output.write(pdf_bytes)
    pdf_output.seek(0)
    return pdf_output

Overwriting report_generator.py


In [19]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

from io import BytesIO


def generate_pdf(content):

    buffer = BytesIO()

    doc = SimpleDocTemplate(buffer)

    styles = getSampleStyleSheet()

    elements = []

    elements.append(
        Paragraph(
            "MeetMind AI Report",
            styles["Title"]
        )
    )

    elements.append(Spacer(1,12))

    for line in content.split("\n"):

        elements.append(
            Paragraph(
                line,
                styles["BodyText"]
            )
        )

    doc.build(elements)

    buffer.seek(0)

    return buffer

In [22]:
!mkdir -p .streamlit

In [23]:
%%writefile .streamlit/config.toml

[theme]
primaryColor="#4F46E5"
backgroundColor="#FFFFFF"
secondaryBackgroundColor="#F3F4F6"
textColor="#111827"
font="sans serif"

Writing .streamlit/config.toml


In [24]:
!ls -la .streamlit

total 12
drwxr-xr-x 2 root root 4096 Jun 14 23:15 .
drwxr-xr-x 1 root root 4096 Jun 14 23:15 ..
-rw-r--r-- 1 root root  131 Jun 14 23:15 config.toml


KPI Cards


In [25]:
col1,col2,col3 = st.columns(3)

col1.metric(
    "Meetings Analyzed",
    "125"
)

col2.metric(
    "Tasks Extracted",
    "578"
)

col3.metric(
    "Time Saved",
    "310 Hours"
)

2026-06-14 23:16:11.100 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.105 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.110 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.116 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:11.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

Analytics Chart

In [26]:
import pandas as pd
import plotly.express as px

data = pd.DataFrame({
    "Category":["Summary","Tasks","Owners"],
    "Count":[1,10,4]
})

fig = px.bar(
    data,
    x="Category",
    y="Count"
)

st.plotly_chart(fig)

2026-06-14 23:16:30.627 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:30.628 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:30.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:30.632 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 23:16:30.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()